In [1]:
from rapidsegment import StrategicSegmentBuilder, StrategicSegmentScore, UniversalDataLoader

# Example Feature Groupings Setup
variable_groups = {
    "Demographics": [
        "age", 
        "gender", 
        "education_level", 
        "dependents", 
        "home_ownership", 
        "city_tier"
    ],
    "Employment": [
        "employment_type", 
        "years_employed"
    ],
    "Financial_Profile": [
        "annual_income", 
        "credit_score", 
        "existing_debt", 
        "account_balance", 
        "monthly_spend", 
        "loan_to_income_ratio", 
        "previous_defaults", 
        "number_of_cards"
    ],
    "Application_Details": [
        "requested_limit", 
        "application_channel"
    ]
}

# 2. Define your dynamically scaled optimization grid
my_search_grid = {
    "min_sample_size": [10000,5000, 1000, 500, 100],  # Example values for min_sample_size   
    "min_lift": [2.0,1.5]
}


# Initialize the builder
builder = StrategicSegmentBuilder(
    target="is_approved",
    # n_jobs=-1,
    min_sample_size=1000,
    min_lift=1.1,
    min_events = 5,
    top_n_vars=15,
    max_segments=10,
    enable_diversity=True,
    max_feature_reuse=1,
    enable_1way=True,  # Include 1-way rules from final output
    enable_2way=True,  # Exclude 2-way rules from final output
    enable_3way=True,  # Exclude 3-way rules from final output
    feature_groups=variable_groups,
    ignore_features=['application_id'],
    param_grid=my_search_grid
)


data = UniversalDataLoader(file_path=r"/workspaces/RapidSegment/credit_app_data.parquet").load()

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)

2026-07-17 16:05:25,707 | INFO     | [builder.py:329] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-17 16:05:26,770 | INFO     | [builder.py:121] | Feature group validation passed. (18 features mapped)
2026-07-17 16:05:26,773 | INFO     | [builder.py:363] | Iteration 1 | Remaining Volume: 500,000 | Base Rate: 26.72%
2026-07-17 16:05:36,054 | INFO     | [builder.py:514] | Feature Usage Tracker Update -> 'loan_to_income_ratio' used count = 1
2026-07-17 16:05:36,054 | INFO     | [builder.py:530] | Segment 1 Captured (Size Floor: 10000 | Lift Floor: 2.0): rows=43878, events=30559.0, lift=2.61
  Rule: loan_to_income_ratio=(-inf, 0.07)
  SQL: loan_to_income_ratio < 0.07
2026-07-17 16:05:37,394 | INFO     | [builder.py:363] | Iteration 2 | Remaining Volume: 456,122 | Base Rate: 22.60%
2026-07-17 16:05:45,515 | INFO     | [builder.py:514] | Feature Usage Tracker Update -> 'credit_score' used count = 1
2026-07-17 16:05:45,516 | INFO     | [builder.py:530] | Segment 2 Captured (Size 

In [2]:
segments_df

[{'segment_id': 1,
  'rule_string': 'loan_to_income_ratio=(-inf, 0.07)',
  'sql_filter': 'loan_to_income_ratio < 0.07',
  'count': 43878,
  'rate': 69.645380372852,
  'lift': 2.606059644850848,
  'meta_applied_sample_size': 10000,
  'meta_applied_min_lift': 2.0},
 {'segment_id': 2,
  'rule_string': 'credit_score=[806.50, inf)',
  'sql_filter': 'credit_score >= 806.50',
  'count': 35515,
  'rate': 54.909193298606226,
  'lift': 2.0546464391569588,
  'meta_applied_sample_size': 10000,
  'meta_applied_min_lift': 2.0}]

In [3]:
from prettytable import PrettyTable
import pandas as pd
table = PrettyTable()
table.field_names = list(pd.DataFrame(final_eval).columns)
for _, row in pd.DataFrame(final_eval).iterrows():
    table.add_row(list(row))
print(table)

+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
| segment | total_count | target_events |   response_rate    | base_response_rate |    capture_rate   |        lift        |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+
|   0.0   |   420607.0  |    83562.0    | 19.86700173796442  |      26.7244       | 84.12140000000001 | 0.7434030974676482 |
|   1.0   |   43878.0   |    30559.0    |  69.645380372852   |      26.7244       |       8.7756      | 2.606059644850848  |
|   2.0   |   35515.0   |    19501.0    | 54.909193298606226 |      26.7244       |       7.103       | 2.0546464391569588 |
+---------+-------------+---------------+--------------------+--------------------+-------------------+--------------------+


In [4]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   loan_to_income_ratio=(-inf, 0.07)
SQL Filter: loan_to_income_ratio < 0.07
--------------------------------------------------
Segment ID: 2
Raw Rule:   credit_score=[806.50, inf)
SQL Filter: credit_score >= 806.50
--------------------------------------------------


In [5]:
builder.explain_feature_journey("city_tier")

AUDIT TRAIL FOR FEATURE: 'city_tier'

[Iteration 1]
  • Current dynamic IV   : 0.0012
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : loan_to_income_ratio=(-inf, 0.07) (Variables: ['loan_to_income_ratio'])

[Iteration 2]
  • Current dynamic IV   : 0.0028
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : credit_score=[806.50, inf) (Variables: ['credit_score'])

[Iteration 3]
  • Current dynamic IV   : 0.0017
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search


In [6]:
# Preparing the dataset for scoring and decile banding.
import duckdb
conn = duckdb.connect()
predicted = conn.register("predicted", data)
predicted = conn.query("""
            SELECT *, 
            CASE WHEN loan_to_income_ratio < 0.08
            THEN 1 ELSE 0 END AS seg_1,
            CASE WHEN credit_score >= 821.50
            THEN 1 ELSE 0 END AS seg_2
            FROM predicted
""").df()
conn.close()

In [7]:
scorer = StrategicSegmentScore(
    target_col="is_approved",
    primary_key="application_id",
    segment_cols=["seg_1","seg_2"],
)

In [8]:
scorer.calculate_and_export_weights(predicted)

2026-07-17 16:05:57,854 | INFO     | [scorer.py:50] | Initializing out-of-core DuckDB scorecard engine...
2026-07-17 16:05:59,555 | INFO     | [scorer.py:88] | Computing harmonic scorecard weights...
2026-07-17 16:05:59,556 | INFO     | [scorer.py:117] | Scoring 100M rows natively via C++ database engine...
2026-07-17 16:05:59,873 | INFO     | [scorer.py:132] | Dataset Zero-Inflation Rate: 73.28%
2026-07-17 16:05:59,874 | INFO     | [scorer.py:135] | Calibrating deciles across active populations...


{'model_metadata': {'total_training_population': 500000,
  'active_scored_population': 500000,
  'active_population_pct': 100.0,
  'baseline_event_rate': 0.2672},
 'segment_weights': {'seg_1': {'weight': 97,
   'lift': 2.5803,
   'response_rate': 0.6896,
   'capture_rate': 0.2582},
  'seg_2': {'weight': 42,
   'lift': 2.2331,
   'response_rate': 0.5968,
   'capture_rate': 0.1129}},
 'decile_min_thresholds': {'1': 139,
  '2': 97,
  '3': 0,
  '4': 0,
  '5': 0,
  '6': 0,
  '7': 0,
  '8': 0,
  '9': 0,
  '10': 0}}

In [9]:
import logging, json
scorecard_json_path = "/workspaces/RapidSegment/scored_experiment_2026_07_13_15_26_02.json"
logging.info(f"Loading scorecard model artifact from {scorecard_json_path}...")
with open(scorecard_json_path, "r") as json_file:
    model_artifact = json.load(json_file)

2026-07-17 16:05:59,967 | INFO     | [409066235.py:3] | Loading scorecard model artifact from /workspaces/RapidSegment/scored_experiment_2026_07_13_15_26_02.json...


FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/RapidSegment/scored_experiment_2026_07_13_15_26_02.json'

In [ ]:
model_artifact.get("decile_min_thresholds")

{'1': 140,
 '2': 98,
 '3': 0,
 '4': 0,
 '5': 0,
 '6': 0,
 '7': 0,
 '8': 0,
 '9': 0,
 '10': 0}

In [ ]:
for key, value in model_artifact.get("segment_weights").items():
    print(f"Segment: {key} | Weight: {value['weight']}")

Segment: seg_1 | Weight: 98
Segment: seg_2 | Weight: 42


In [ ]:
conn = duckdb.connect()
scored = conn.register("scored", predicted)
scored = conn.query("""
WITH CTE AS (
    SELECT *, 
    CASE WHEN seg_1 = 1 THEN 98 ELSE 0 END AS seg_1_weighted,
    CASE WHEN seg_2 = 1 THEN 42 ELSE 0 END AS seg_2_weighted
    FROM scored),
    CTE2 AS (
    SELECT *, (seg_1_weighted + seg_2_weighted) AS total_weight
                     FROM CTE)
SELECT *, CASE WHEN total_weight >=140 THEN 1
                    WHEN total_weight >= 98 THEN 2
                    WHEN total_weight >= 0 THEN 3
                    WHEN total_weight >= 0 THEN 4
                    WHEN total_weight >= 0 THEN 5
                    WHEN total_weight >= 0 THEN 6
                    WHEN total_weight >= 0 THEN 7
                    WHEN total_weight >= 0 THEN 8
                    WHEN total_weight >= 0 THEN 9
                    WHEN total_weight >= 0 THEN 10
                    ELSE 0 END AS decile_band
                    
                     FROM CTE2
""").to_df()
conn.close()

In [ ]:
conn = duckdb.connect()
scored = conn.register("scored", scored)
scored = conn.query("""SELECT decile_band, 
                    COUNT(*) AS count, 
                    SUM(is_approved) AS events, 
                    (SUM(is_approved)*100.0/COUNT(*)) AS response_rate
FROM scored
GROUP BY decile_band
ORDER BY decile_band
""").to_df()
conn.close()
scored

,decile_band,count,events,response_rate
0,1,2531,2504.0,98.933228
1,2,47706,32235.0,67.570117
2,3,449763,98699.0,21.944669
